# TracknetV1 — Validação de Pós-processamento

Ajuste os hiperparâmetros, execute as células em ordem e avalie o dataset de validação inteiro.

> A função `validate` abaixo replica a mesma regra de validação usada no treino.


In [ ]:
# ===== CAMINHOS E EXECUÇÃO =====
MODEL_WEIGHTS_PATH = 'assets/model.pt'
VAL_PATH = 'assets/val'

BATCH_SIZE = 1
MAX_VAL_SAMPLES = None  # None = dataset inteiro
DEVICE = 'cuda'  # usa CPU automaticamente se CUDA não estiver disponível

# ===== HIPERPARÂMETROS =====
THRESHOLD = 205
DP = 1.0
HOUGH_MIN_DIST = 1
PARAM1 = 116
PARAM2 = 10
MIN_RADIUS = 2
MAX_RADIUS = 7

# Distância máxima, em pixels da imagem original, para um TP
MATCH_DISTANCE_PX = 30


In [ ]:
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import distance
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from datasets.TracknetV1Dataset import TracknetV1Dataset
from models.TracknetV1 import BallTrackerNet


In [ ]:
device = 'cuda' if DEVICE == 'cuda' and torch.cuda.is_available() else 'cpu'

val_dataset = TracknetV1Dataset(
    frames_path=os.path.join(VAL_PATH, 'frames_out'),
    gts_path=os.path.join(VAL_PATH, 'gts'),
    labels_path=os.path.join(VAL_PATH, 'labels'),
)
if MAX_VAL_SAMPLES is not None:
    val_dataset = Subset(val_dataset, range(min(MAX_VAL_SAMPLES, len(val_dataset))))

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = BallTrackerNet().to(device)
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model.eval()

HEATMAP_H, HEATMAP_W = 360, 640

print(f'Device: {device} | amostras: {len(val_dataset)}')


In [ ]:
def postprocess(feature_map, scale=2):
    feature_map = feature_map.copy()
    feature_map *= 255
    feature_map = feature_map.reshape((HEATMAP_H, HEATMAP_W))
    feature_map = feature_map.astype(np.uint8)
    _, heatmap = cv2.threshold(feature_map, THRESHOLD, 255, cv2.THRESH_BINARY)
    circles = cv2.HoughCircles(
        heatmap, cv2.HOUGH_GRADIENT,
        dp=DP, minDist=HOUGH_MIN_DIST,
        param1=PARAM1, param2=PARAM2,
        minRadius=MIN_RADIUS, maxRadius=MAX_RADIUS,
    )
    x, y = None, None
    if circles is not None:
        if len(circles) == 1:
            x = circles[0][0][0] * scale
            y = circles[0][0][1] * scale
    return x, y


def validate(model, val_loader, device, epoch, min_dist=30):
    losses = []
    tp = [0, 0, 0, 0]
    fp = [0, 0, 0, 0]
    tn = [0, 0, 0, 0]
    fn = [0, 0, 0, 0]
    criterion = nn.CrossEntropyLoss()
    model.eval()
    for iter_id, batch in enumerate(val_loader):
        with torch.no_grad():
            out = model(batch[0].float().to(device))
            gt = torch.tensor(batch[1], dtype=torch.long, device=device)
            loss = criterion(out, gt)
            losses.append(loss.item())
            output = out.argmax(dim=1).detach().cpu().numpy()
            for i in range(len(output)):
                x_pred, y_pred = postprocess(output[i])
                x_gt = batch[2][i]
                y_gt = batch[3][i]
                vis = batch[4][i]
                if x_pred:
                    if vis != 0:
                        dst = distance.euclidean((x_pred, y_pred), (x_gt, y_gt))
                        if dst < min_dist:
                            tp[vis] += 1
                        else:
                            fp[vis] += 1
                    else:
                        fp[vis] += 1
                if not x_pred:
                    if vis != 0:
                        fn[vis] += 1
                    else:
                        tn[vis] += 1
            print('val | epoch = {}, iter = [{}|{}], loss = {}, tp = {}, tn = {}, fp = {}, fn = {} '.format(
                epoch, iter_id, len(val_loader), round(np.mean(losses), 6),
                sum(tp), sum(tn), sum(fp), sum(fn)))
    eps = 1e-15
    precision = sum(tp) / (sum(tp) + sum(fp) + eps)
    vc1 = tp[1] + fp[1] + tn[1] + fn[1]
    vc2 = tp[2] + fp[2] + tn[2] + fn[2]
    vc3 = tp[3] + fp[3] + tn[3] + fn[3]
    recall = sum(tp) / (vc1 + vc2 + vc3 + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    print('precision = {}'.format(precision))
    print('recall = {}'.format(recall))
    print('f1 = {}'.format(f1))

    global LAST_COUNTS
    LAST_COUNTS = {'TP': sum(tp), 'TN': sum(tn), 'FP': sum(fp), 'FN': sum(fn)}
    return np.mean(losses), precision, recall, f1


In [ ]:
avg_loss, precision, recall, f1 = validate(model, val_loader, device, epoch=0, min_dist=MATCH_DISTANCE_PX)
counts = LAST_COUNTS

print(f'Hiperparâmetros: thr={THRESHOLD}, dp={DP}, minDist={HOUGH_MIN_DIST}, p1={PARAM1}, p2={PARAM2}, r=[{MIN_RADIUS}, {MAX_RADIUS}]')
print(f'Loss={avg_loss:.6f} | Precision={precision:.4f} | Recall={recall:.4f} | F1={f1:.4f}')
print(f"TP={counts['TP']} TN={counts['TN']} FP={counts['FP']} FN={counts['FN']}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ['TP', 'TN', 'FP', 'FN']
values = [counts[label] for label in labels]
bars = axes[0].bar(labels, values, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
axes[0].bar_label(bars)
axes[0].set_title('Contagens da validação')
axes[0].grid(axis='y', alpha=.25)

metric_bars = axes[1].bar(['Precision', 'Recall', 'F1'], [precision, recall, f1], color=['#9b59b6', '#1abc9c', '#e67e22'])
axes[1].bar_label(metric_bars, fmt='%.3f')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Métricas')
axes[1].grid(axis='y', alpha=.25)

plt.tight_layout()
plt.show()
